# Python Course — Part 3

**Covers:** Section 7 — Advanced Python | Section 8 — Standard Library | Section 9 — Expert Insights | Section 10 — Case Studies

---

---
# Section 7 — Advanced Python

## 7.1 Metaclasses

### Concept

A **metaclass** is the class of a class. Just as objects are instances of classes, **classes are instances of metaclasses**.  
The default metaclass is `type`. Metaclasses let you intercept and modify class creation — adding methods, enforcing contracts, registering subclasses automatically.

### Technical Deep Dive

When Python executes `class Foo(Base): ...`, it calls `type.__new__(mcs, name, bases, namespace)`.  
You override this by setting `metaclass=MyMeta` in the class definition.  
The call chain: `type.__call__` → `MyMeta.__new__` → `MyMeta.__init__`.

Use metaclasses sparingly — class decorators and `__init_subclass__` solve most use cases more simply.

In [ ]:
# type() — the default metaclass
print(type(int))        # <class 'type'>
print(type(list))       # <class 'type'>
print(type(type))       # <class 'type'>  — type is its own metaclass

# Dynamically create a class with type()
# type(name, bases, namespace)
MyClass = type("MyClass", (object,), {
    "greeting": "Hello",
    "greet": lambda self: f"{self.greeting}, I am {self.__class__.__name__}"
})

obj = MyClass()
print(obj.greet())

In [ ]:
# Custom metaclass — enforce naming conventions
class SnakeCaseMeta(type):
    def __new__(mcs, name, bases, namespace):
        for attr_name in namespace:
            if not attr_name.startswith("_") and attr_name != attr_name.lower():
                raise TypeError(
                    f"Attribute '{attr_name}' in class '{name}' must be snake_case"
                )
        return super().__new__(mcs, name, bases, namespace)

class GoodClass(metaclass=SnakeCaseMeta):
    my_value = 42
    def compute_result(self):
        return self.my_value * 2

try:
    class BadClass(metaclass=SnakeCaseMeta):
        myValue = 42    # camelCase — will be rejected
except TypeError as e:
    print(e)

g = GoodClass()
print(g.compute_result())

In [ ]:
# __init_subclass__ — simpler alternative for most metaclass use cases
class PluginBase:
    _registry: dict = {}

    def __init_subclass__(cls, plugin_name: str = "", **kwargs):
        super().__init_subclass__(**kwargs)
        if plugin_name:
            PluginBase._registry[plugin_name] = cls
        print(f"Registered plugin: {plugin_name!r} -> {cls.__name__}")

class CSVPlugin(PluginBase, plugin_name="csv"):
    def process(self):
        return "processing CSV"

class JSONPlugin(PluginBase, plugin_name="json"):
    def process(self):
        return "processing JSON"

print(PluginBase._registry)
plugin = PluginBase._registry["csv"]()
print(plugin.process())

## 7.2 Descriptors

### Concept

A **descriptor** is an object that defines `__get__`, `__set__`, and/or `__delete__` — controlling attribute access on another class.  
This is how `@property`, `@classmethod`, `@staticmethod`, and ORM fields like Django's `CharField` are implemented.

### Technical Deep Dive

- **Data descriptor**: defines `__set__` (or `__delete__`) — takes priority over instance `__dict__`.
- **Non-data descriptor**: defines only `__get__` — instance `__dict__` takes priority.
- The lookup chain: data descriptor → instance `__dict__` → non-data descriptor → class `__dict__`.

In [ ]:
# Typed and validated descriptor
class TypedField:
    def __init__(self, name, expected_type, min_val=None, max_val=None):
        self.name = name
        self.expected_type = expected_type
        self.min_val = min_val
        self.max_val = max_val

    def __set_name__(self, owner, name):  # called when class is defined
        self.name = name
        self.private = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self   # accessed on class, return descriptor itself
        return getattr(obj, self.private, None)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name}: expected {self.expected_type.__name__}, got {type(value).__name__}")
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f"{self.name}: must be >= {self.min_val}")
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f"{self.name}: must be <= {self.max_val}")
        setattr(obj, self.private, value)


class Person:
    name = TypedField("name", str)
    age  = TypedField("age", int, min_val=0, max_val=150)

    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"Person(name={self.name!r}, age={self.age})"


p = Person("Alice", 30)
print(p)

try:
    p.age = -5
except ValueError as e:
    print(e)

try:
    p.name = 123
except TypeError as e:
    print(e)

## 7.3 `__slots__` — Memory Optimization

### Concept

By default, Python stores instance attributes in a per-instance `__dict__`, which has overhead.  
`__slots__` tells Python to allocate a fixed-size array instead — **reducing memory usage by 40-50%** for classes with many instances and preventing dynamic attribute creation.

In [ ]:
import sys

class PointRegular:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointSlotted:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

r = PointRegular(1, 2)
s = PointSlotted(1, 2)

print(f"Regular:  {sys.getsizeof(r)} bytes, has __dict__: {hasattr(r, '__dict__')}")
print(f"Slotted:  {sys.getsizeof(s)} bytes, has __dict__: {hasattr(s, '__dict__')}")

# Cannot add dynamic attributes to slotted class
try:
    s.z = 3
except AttributeError as e:
    print(e)

# Benchmark: 1 million instances
import tracemalloc

tracemalloc.start()
regular_pts = [PointRegular(i, i) for i in range(100_000)]
_, regular_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
slotted_pts = [PointSlotted(i, i) for i in range(100_000)]
_, slotted_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Regular peak: {regular_peak/1024:.1f} KB")
print(f"Slotted peak: {slotted_peak/1024:.1f} KB")
print(f"Savings: {(1 - slotted_peak/regular_peak)*100:.1f}%")

## 7.4 Functional Programming — `functools` and `itertools`

### Concept

`functools` and `itertools` are the standard library's functional programming toolkit.  
`itertools` provides composable, memory-efficient iterators inspired by Haskell and APL.

In [ ]:
import itertools
import functools

# itertools — infinite iterators
counter = itertools.count(10, 2)          # 10, 12, 14, ...
cycler  = itertools.cycle(["R", "G", "B"]) # R G B R G B ...
repeater = itertools.repeat("x", 3)       # x x x

print(list(itertools.islice(counter, 5)))  # first 5: [10, 12, 14, 16, 18]
print(list(itertools.islice(cycler, 7)))   # [R G B R G B R]
print(list(repeater))

# Combinatoric iterators
items = ["A", "B", "C"]
print("Permutations:", list(itertools.permutations(items, 2)))
print("Combinations:", list(itertools.combinations(items, 2)))
print("Product:",      list(itertools.product([0,1], repeat=3)))

In [ ]:
# groupby — group consecutive items by key (data must be sorted by key first!)
data = [
    {"name": "Alice", "dept": "Eng"},
    {"name": "Bob",   "dept": "Eng"},
    {"name": "Carol", "dept": "HR"},
    {"name": "Dave",  "dept": "HR"},
    {"name": "Eve",   "dept": "Eng"},
]

sorted_data = sorted(data, key=lambda x: x["dept"])
for dept, members in itertools.groupby(sorted_data, key=lambda x: x["dept"]):
    names = [m["name"] for m in members]
    print(f"{dept}: {names}")

# chain, chain.from_iterable
nested = [[1, 2], [3, 4], [5, 6]]
flat = list(itertools.chain.from_iterable(nested))
print(flat)

# accumulate — running totals
import operator
sales = [100, 200, 150, 300, 250]
running_total = list(itertools.accumulate(sales))
running_max   = list(itertools.accumulate(sales, func=max))
print("Running total:", running_total)
print("Running max:  ", running_max)

## 7.5 Concurrency — Threading

### Concept

Python threads share memory and are managed by the OS.  
The **Global Interpreter Lock (GIL)** prevents true parallel execution of Python bytecode — threads excel at **I/O-bound** tasks (network, disk) but not CPU-bound ones.

### Technical Deep Dive

- Use `threading.Thread` for simple cases.
- Use `concurrent.futures.ThreadPoolExecutor` for pools of threads with result handling.
- Shared mutable state requires `threading.Lock` to avoid race conditions.

In [ ]:
import threading
import time
import concurrent.futures

# ThreadPoolExecutor — I/O-bound simulation
def fetch_data(url_id):
    time.sleep(0.1)   # simulate network latency
    return f"data_from_{url_id}"

ids = list(range(10))

start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch_data, ids))
elapsed = time.perf_counter() - start

print(f"Fetched {len(results)} items in {elapsed:.2f}s")
print(results[:3])

In [ ]:
# Race condition and Lock
import threading

counter = 0
lock = threading.Lock()

def increment_safe(n):
    global counter
    for _ in range(n):
        with lock:     # acquire lock; released even on exception
            counter += 1

threads = [threading.Thread(target=increment_safe, args=(10_000,)) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Counter: {counter} (expected {5 * 10_000})")

## 7.6 Concurrency — Multiprocessing

### Concept

**Multiprocessing** bypasses the GIL by spawning separate processes — each with its own Python interpreter and memory space.  
Use for **CPU-bound** tasks: number crunching, image processing, ML training loops.

### Technical Deep Dive

- `ProcessPoolExecutor` mirrors `ThreadPoolExecutor` — swap one line to switch.
- Inter-process communication via `Queue`, `Pipe`, `Value`, `Array` (shared memory).
- Pickling overhead: arguments and results are serialized — keep them small.

In [ ]:
import concurrent.futures
import time

def cpu_heavy(n):
    """Simulate CPU-bound work."""
    return sum(i * i for i in range(n))

tasks = [10_000_000] * 4

# Sequential
start = time.perf_counter()
seq_results = [cpu_heavy(n) for n in tasks]
seq_time = time.perf_counter() - start

# Parallel (ProcessPoolExecutor)
start = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor() as executor:
    par_results = list(executor.map(cpu_heavy, tasks))
par_time = time.perf_counter() - start

print(f"Sequential:  {seq_time:.2f}s")
print(f"Parallel:    {par_time:.2f}s")
print(f"Speedup:     {seq_time/par_time:.2f}x")
print(f"Results match: {seq_results == par_results}")

## 7.7 Asynchronous Programming — `asyncio`

### Concept

`asyncio` implements **cooperative multitasking** in a single thread using an **event loop**.  
Coroutines (`async def`) yield control with `await` — letting the event loop run other coroutines while waiting for I/O.

### Technical Deep Dive

- `async def` defines a coroutine — calling it returns a coroutine object, not the result.
- `await` suspends the current coroutine until the awaitable completes.
- `asyncio.gather()` runs multiple coroutines concurrently.
- Use `asyncio.run()` as the single entry point for async programs.
- Perfect for network I/O with `aiohttp`, `httpx`, database with `asyncpg`, etc.

In [ ]:
import asyncio
import time

async def fetch(url_id: int, delay: float) -> str:
    print(f"  Start fetch {url_id}")
    await asyncio.sleep(delay)   # non-blocking sleep
    print(f"  Done  fetch {url_id}")
    return f"result_{url_id}"

async def main():
    start = time.perf_counter()

    # gather runs all concurrently
    results = await asyncio.gather(
        fetch(1, 0.3),
        fetch(2, 0.1),
        fetch(3, 0.2),
    )

    elapsed = time.perf_counter() - start
    print(f"All done in {elapsed:.2f}s — results: {results}")

# asyncio.run() is the entry point
asyncio.run(main())

In [ ]:
# Async context managers and iterators
import asyncio

class AsyncTimer:
    async def __aenter__(self):
        self._start = asyncio.get_event_loop().time()
        return self

    async def __aexit__(self, *args):
        elapsed = asyncio.get_event_loop().time() - self._start
        print(f"Async block took {elapsed:.3f}s")

async def async_generator(n):
    for i in range(n):
        await asyncio.sleep(0.01)
        yield i

async def demo_async_features():
    async with AsyncTimer():
        async for value in async_generator(5):
            print(value, end=" ")
        print()

    # asyncio.create_task — fire and forget
    task = asyncio.create_task(fetch(99, 0.05))
    print("Task created, doing other work...")
    result = await task
    print(result)

asyncio.run(demo_async_features())

## 7.8 Memory Management and Garbage Collection

### Concept

Python manages memory through **reference counting** (primary) and a **cyclic garbage collector** (secondary, handles reference cycles).  
Understanding this is critical for writing memory-efficient code and avoiding leaks.

In [ ]:
import gc
import sys
import weakref

# Reference counting
a = [1, 2, 3]
b = a            # ref count = 2
c = a            # ref count = 3
print(sys.getrefcount(a))   # 4 (includes getrefcount's own arg)

del b            # ref count = 3
del c            # ref count = 2
# when ref count reaches 0, object is immediately freed

# Reference cycles — caught by gc
class Node:
    def __init__(self, value):
        self.value = value
        self.ref = None

n1 = Node(1)
n2 = Node(2)
n1.ref = n2
n2.ref = n1     # cycle: n1 <-> n2
del n1, n2       # ref counts > 0 due to cycle, but gc handles it

collected = gc.collect()
print(f"GC collected {collected} objects")

# Weak references — don't increase ref count
class Cache:
    def __init__(self):
        self._store = weakref.WeakValueDictionary()

    def set(self, key, value):
        self._store[key] = value

    def get(self, key):
        return self._store.get(key)

cache = Cache()
obj = ["important data"]
cache.set("key", obj)
print(cache.get("key"))   # found
del obj
print(cache.get("key"))   # None — object was GC'd

## 7.9 Python Internals — CPython, GIL, Bytecode

### Concept

**CPython** is the reference implementation of Python (written in C).  
Python source is compiled to **bytecode** (`.pyc` files) which is then executed by the CPython VM.  
The **GIL** (Global Interpreter Lock) is a mutex that allows only one thread to execute Python bytecode at a time.

In [ ]:
import dis
import sys

# Inspect bytecode
def add(a, b):
    return a + b

print("=== Bytecode for add() ===")
dis.dis(add)

# Code object internals
code = add.__code__
print(f"\nArguments : {code.co_varnames[:code.co_argcount]}")
print(f"Constants  : {code.co_consts}")
print(f"Stack size : {code.co_stacksize}")

# Python version and implementation
print(f"\nImplementation: {sys.implementation.name}")
print(f"Version: {sys.version_info[:3]}")

# Small integer caching (-5 to 256)
a = 256
b = 256
print(f"\n256 is 256: {a is b}")   # True — cached
a = 257
b = 257
print(f"257 is 257: {a is b}")   # False — not cached (CPython implementation detail)

### Exercises — Section 7

1. Write a metaclass `Singleton` that ensures only one instance of any class using it can be created.
2. Write a descriptor `Clamped` that clamps a numeric value to `[min_val, max_val]` on assignment.
3. Using `asyncio`, write an async function `download_all(urls)` that fetches all URLs concurrently using `asyncio.gather` and returns results with timing info.
4. Write a class `SlottedPoint3D` with `__slots__` and compare memory usage against a regular `Point3D` with 100,000 instances.

### Mini Challenge

Implement a thread-safe `BoundedQueue` class using `threading.Lock` and a `deque`. Support `put(item)` (blocks if full), `get()` (blocks if empty), and `qsize()`. Use `threading.Condition` for blocking.

### Best Practices — Section 7

- Prefer `__init_subclass__` or class decorators over metaclasses — simpler and more readable.
- Use `__slots__` for classes instantiated millions of times (e.g., data records, graph nodes).
- Use `ThreadPoolExecutor` for I/O-bound work; `ProcessPoolExecutor` for CPU-bound work.
- Always `await` coroutines — forgetting returns a coroutine object, not the result.
- Use `asyncio.gather()` for concurrent awaiting; `asyncio.create_task()` for fire-and-forget.

### Common Mistakes — Section 7

- Using threads for CPU-bound tasks — the GIL makes this slower than sequential.
- Calling `asyncio.run()` inside a running event loop — use `await` instead.
- Forgetting `__set_name__` in descriptors — name is not automatically set before Python 3.6.
- Reference cycles with `__del__` — can prevent garbage collection.

### Summary — Section 7

- Metaclasses let you control class creation — `type` is the default; `__init_subclass__` is simpler.
- Descriptors power `@property`, `@classmethod`, ORM fields — `__get__`/`__set__`/`__delete__`.
- `__slots__` reduces memory by 40-50% for data-heavy classes.
- Threads for I/O-bound; processes for CPU-bound; `asyncio` for high-concurrency I/O.
- CPython uses reference counting + cyclic GC. Use `weakref` to avoid cycles in caches.

In [ ]:
# Exercise 1 — Singleton metaclass
class SingletonMeta(type):
    # YOUR CODE HERE
    pass
# Exercise 2 — Clamped descriptor
class Clamped:
    # YOUR CODE HERE
    pass
# Mini Challenge — BoundedQueue
# YOUR CODE HERE

---
# Section 8 — Standard Library Deep Dive

## 8.1 `datetime` and `zoneinfo`

### Concept

Python's `datetime` module handles dates, times, and timedeltas.  
Always work with **timezone-aware** datetimes in production — use `zoneinfo` (Python 3.9+) or `pytz`.

In [ ]:
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo

# Naive vs aware datetime
naive = datetime.now()
aware = datetime.now(tz=timezone.utc)
warsaw = datetime.now(tz=ZoneInfo("Europe/Warsaw"))

print(f"Naive  : {naive}")
print(f"UTC    : {aware}")
print(f"Warsaw : {warsaw}")

# Arithmetic
today = date.today()
next_week  = today + timedelta(weeks=1)
yesterday  = today - timedelta(days=1)
print(f"Today: {today}, Next week: {next_week}, Yesterday: {yesterday}")

# Parsing and formatting
dt = datetime.strptime("2024-03-15 09:30", "%Y-%m-%d %H:%M")
print(dt.strftime("%A, %d %B %Y"))   # Friday, 15 March 2024
print(dt.isoformat())                 # 2024-03-15T09:30:00
print(datetime.fromisoformat("2024-03-15T09:30:00"))

# Convert between timezones
utc_time = datetime(2024, 3, 15, 12, 0, tzinfo=timezone.utc)
ny_time = utc_time.astimezone(ZoneInfo("America/New_York"))
print(f"UTC: {utc_time.strftime('%H:%M')} -> NY: {ny_time.strftime('%H:%M %Z')}")

## 8.2 `logging` — Structured Logging

### Concept

The `logging` module provides a flexible, configurable logging system.  
Never use `print()` for application logs — `logging` supports levels, handlers, formatters, and structured output.

In [ ]:
import logging
import json
import sys

# Basic setup
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("myapp.auth")

logger.debug("Debug message")
logger.info("User logged in")
logger.warning("Token expiring soon")
logger.error("Authentication failed")
logger.critical("Database unreachable")

In [ ]:
# Structured JSON logging
class JsonFormatter(logging.Formatter):
    def format(self, record):
        log_dict = {
            "timestamp": self.formatTime(record),
            "level":     record.levelname,
            "logger":    record.name,
            "message":   record.getMessage(),
        }
        if record.exc_info:
            log_dict["exception"] = self.formatException(record.exc_info)
        if hasattr(record, "extra"):
            log_dict.update(record.extra)
        return json.dumps(log_dict)

json_logger = logging.getLogger("json_app")
json_logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JsonFormatter())
json_logger.addHandler(handler)
json_logger.propagate = False

json_logger.info("Request received")

try:
    1 / 0
except ZeroDivisionError:
    json_logger.error("Division error", exc_info=True)

## 8.3 `unittest` and `pytest`

### Concept

`unittest` is Python's built-in test framework. `pytest` is the de-facto standard — less boilerplate, powerful fixtures, better output.  
Good tests are **isolated**, **deterministic**, and **fast**.

In [ ]:
import unittest

# Function under test
def is_palindrome(s: str) -> bool:
    cleaned = s.lower().replace(" ", "")
    return cleaned == cleaned[::-1]

class TestPalindrome(unittest.TestCase):
    def test_simple_palindrome(self):
        self.assertTrue(is_palindrome("racecar"))

    def test_with_spaces(self):
        self.assertTrue(is_palindrome("race car"))

    def test_not_palindrome(self):
        self.assertFalse(is_palindrome("hello"))

    def test_empty(self):
        self.assertTrue(is_palindrome(""))

    def test_case_insensitive(self):
        self.assertTrue(is_palindrome("Racecar"))

    def test_raises_on_non_string(self):
        with self.assertRaises(AttributeError):
            is_palindrome(123)

# Run in notebook
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestPalindrome)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [ ]:
# pytest-style (no class needed)
# These would normally live in test_palindrome.py and run with `pytest`

import pytest

@pytest.mark.parametrize("text,expected", [
    ("racecar",  True),
    ("hello",    False),
    ("A man a plan a canal Panama", True),
    ("",         True),
])
def test_palindrome_parametrized(text, expected):
    assert is_palindrome(text) == expected

# Mocking with unittest.mock
from unittest.mock import patch, MagicMock

def get_config_value(key):
    import os
    return os.environ.get(key, "default")

with patch("os.environ") as mock_env:
    mock_env.get = MagicMock(return_value="mocked_value")
    result = get_config_value("DB_URL")
    print(f"Mocked result: {result}")
    mock_env.get.assert_called_once_with("DB_URL", "default")

## 8.4 `argparse` — CLI Applications

### Concept

`argparse` parses command-line arguments with automatic help generation, type coercion, and validation.  
It is the standard for building production CLI tools in Python.

In [ ]:
import argparse

def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="data-tool",
        description="Process data files.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
    )

    # Positional argument
    parser.add_argument("input", help="Input file path")

    # Optional arguments
    parser.add_argument("-o", "--output", default="out.csv", help="Output path")
    parser.add_argument("-n", "--rows", type=int, default=100, help="Number of rows")
    parser.add_argument("-v", "--verbose", action="store_true", help="Verbose output")
    parser.add_argument("--format", choices=["csv", "json", "parquet"], default="csv")

    # Subcommands
    subparsers = parser.add_subparsers(dest="command")

    validate = subparsers.add_parser("validate", help="Validate data")
    validate.add_argument("--strict", action="store_true")

    return parser

# Simulate: data-tool input.csv -n 50 --format json -v
parser = build_parser()
args = parser.parse_args(["input.csv", "-n", "50", "--format", "json", "-v"])
print(args)
print(f"Input: {args.input}, Format: {args.format}, Rows: {args.rows}")

## 8.5 `dataclasses` + `pydantic` — Data Validation

### Concept

`pydantic` extends dataclasses with **runtime validation**, coercion, and JSON serialization.  
It is the foundation of FastAPI and is widely used for config management and API contracts.

In [ ]:
# Install if needed: pip install pydantic
try:
    from pydantic import BaseModel, Field, field_validator, ValidationError
    from typing import Optional

    class Address(BaseModel):
        street: str
        city: str
        country: str = "PL"

    class User(BaseModel):
        id: int
        name: str = Field(min_length=2, max_length=100)
        email: str
        age: Optional[int] = Field(None, ge=0, le=150)
        address: Optional[Address] = None

        @field_validator("email")
        @classmethod
        def email_must_have_at(cls, v):
            if "@" not in v:
                raise ValueError("Invalid email")
            return v.lower()

    user = User(
        id="42",           # auto-coerced to int
        name="Alice",
        email="ALICE@Example.COM",
        age=30,
        address={"street": "ul. Nowa 1", "city": "Warsaw"}
    )
    print(user)
    print(user.model_dump_json(indent=2))

    # Validation error
    try:
        User(id=1, name="X", email="bad-email", age=-1)
    except ValidationError as e:
        print(e)

except ImportError:
    print("pydantic not installed. Run: pip install pydantic")

## 8.6 `subprocess` — Running Shell Commands

### Concept

`subprocess` lets Python launch and interact with external processes.  
Always prefer `subprocess.run()` over `os.system()` — it captures output and raises on failure.

In [ ]:
import subprocess

# Basic run
result = subprocess.run(
    ["python3", "--version"],
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
print(f"Return code: {result.returncode}")

# Check=True raises CalledProcessError on non-zero exit
try:
    subprocess.run(["ls", "/nonexistent"], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print(f"Failed: {e.returncode} | {e.stderr.strip()}")

# Pipe between commands
ps = subprocess.run(["echo", "hello world"], capture_output=True, text=True)
wc = subprocess.run(["wc", "-w"], input=ps.stdout, capture_output=True, text=True)
print(f"Word count: {wc.stdout.strip()}")

## 8.7 `pickle` and Serialization

### Concept

`pickle` serializes arbitrary Python objects to bytes.  
Use for caching, inter-process communication, and saving model state.  
Never unpickle data from untrusted sources — it can execute arbitrary code.

In [ ]:
import pickle
from pathlib import Path

# Pickle any Python object
data = {
    "model_weights": [0.1, 0.5, -0.3],
    "config": {"lr": 0.001, "epochs": 100},
    "transform": lambda x: x * 2,   # even lambdas
}

path = Path("/tmp/model.pkl")

# Serialize
with path.open("wb") as f:
    pickle.dump(data, f)

# Deserialize
with path.open("rb") as f:
    loaded = pickle.load(f)

print(loaded["config"])
print(loaded["transform"](5))   # 10

# Custom pickle support
class MyModel:
    def __init__(self, weights):
        self.weights = weights
        self._cache = {}  # don't pickle this

    def __getstate__(self):
        state = self.__dict__.copy()
        del state["_cache"]
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._cache = {}  # reinitialize

m = MyModel([1, 2, 3])
m._cache["x"] = "computed"
m2 = pickle.loads(pickle.dumps(m))
print(m2.weights, m2._cache)   # cache is empty after unpickling

### Exercises — Section 8

1. Write a CLI tool using `argparse` that accepts a CSV file path, a column name, and an operation (`sum`, `mean`, `max`) and prints the result.
2. Write a `setup_logging(level, log_file)` function that configures a logger that writes to both stdout and a rotating file (use `logging.handlers.RotatingFileHandler`).
3. Write a `pydantic` model `Config` that loads from environment variables using `pydantic-settings` or reads a JSON config file, with fields `db_url`, `port` (default 8080), `debug` (bool, default False).

### Mini Challenge

Build a `FileCache` class that stores function call results on disk using `pickle`, keyed by function name and arguments. Implement it as a decorator `@file_cache(directory)`.

### Solutions

In [ ]:
# Exercise 2 — setup_logging with RotatingFileHandler
# YOUR CODE HERE
# Mini Challenge — @file_cache decorator
# YOUR CODE HERE

---
# Section 9 — Expert-Level Insights

## 9.1 Writing Pythonic Code

### Concept

**Pythonic code** is idiomatic — it reads naturally, uses language features as intended, and avoids unnecessary complexity.  
The Zen of Python: "There should be one — and preferably only one — obvious way to do it."

In [ ]:
# Anti-patterns vs Pythonic equivalents

items = [1, 2, 3, 4, 5]

# BAD: C-style index loop
for i in range(len(items)):
    print(items[i])

# GOOD: iterate directly
for item in items:
    print(item)

# BAD: manual index tracking
i = 0
for item in items:
    print(i, item)
    i += 1

# GOOD: enumerate
for i, item in enumerate(items):
    print(i, item)

# BAD: building dict manually
keys = ["a", "b", "c"]
values = [1, 2, 3]
d = {}
for k, v in zip(keys, values):
    d[k] = v

# GOOD: dict comprehension or dict(zip())
d = dict(zip(keys, values))

# BAD: checking None
x = None
if x == None: pass

# GOOD: identity check
if x is None: pass

print("All pythonic idioms demonstrated")

In [ ]:
# More Pythonic patterns

# EAFP (Easier to Ask Forgiveness than Permission) — preferred in Python
def get_value_eafp(d, key):
    try:
        return d[key]
    except KeyError:
        return None

# LBYL (Look Before You Leap) — less Pythonic
def get_value_lbyl(d, key):
    if key in d:
        return d[key]
    return None

# BEST: use .get()
d = {"a": 1}
val = d.get("b", None)

# Context variables as self-documenting code
MAX_RETRIES = 3
DEFAULT_TIMEOUT_SEC = 30
ALLOWED_EXTENSIONS = frozenset({".csv", ".json", ".parquet"})

# Walrus operator (Python 3.8+) — assign and test in one
import re
text = "Error 404: Not Found"
if m := re.search(r"(\d{3})", text):
    print(f"Found status code: {m.group(1)}")

# any() / all() with generator expressions
numbers = [2, 4, 6, 8, 10]
print(all(n % 2 == 0 for n in numbers))  # True — all even
print(any(n > 9 for n in numbers))        # True — any > 9

## 9.2 Performance Profiling

### Concept

Premature optimization is the root of all evil. **Measure first, optimize second.**  
Tools: `timeit` (microbenchmarks), `cProfile` (function-level), `line_profiler` (line-level), `memory_profiler`.

In [ ]:
import timeit
import cProfile
import pstats
import io

# timeit — microbenchmarks
def join_with_plus(n):
    result = ""
    for i in range(n):
        result += str(i)
    return result

def join_with_join(n):
    return "".join(str(i) for i in range(n))

n = 1000
t_plus = timeit.timeit(lambda: join_with_plus(n), number=500)
t_join = timeit.timeit(lambda: join_with_join(n), number=500)

print(f"str +=    : {t_plus*1000:.2f}ms")
print(f"str.join  : {t_join*1000:.2f}ms")
print(f"join is {t_plus/t_join:.1f}x faster")

In [ ]:
import cProfile, pstats, io

def slow_function():
    total = 0
    for i in range(100_000):
        total += sum(range(i % 10))
    return total

# Profile with cProfile
pr = cProfile.Profile()
pr.enable()
result = slow_function()
pr.disable()

stream = io.StringIO()
stats = pstats.Stats(pr, stream=stream)
stats.sort_stats("cumulative")
stats.print_stats(5)   # top 5 by cumulative time
print(stream.getvalue())

## 9.3 Design Patterns in Python

### Concept

Design patterns are reusable solutions to common problems. Python's first-class functions and dynamic typing often simplify classical Gang-of-Four patterns significantly.

In [ ]:
# 1. Factory Pattern
class CSVReader:
    def read(self, path): return f"Reading CSV: {path}"

class JSONReader:
    def read(self, path): return f"Reading JSON: {path}"

def reader_factory(fmt: str):
    readers = {"csv": CSVReader, "json": JSONReader}
    if fmt not in readers:
        raise ValueError(f"Unknown format: {fmt}")
    return readers[fmt]()

r = reader_factory("json")
print(r.read("data.json"))

In [ ]:
# 2. Observer Pattern
from typing import Callable, List

class EventEmitter:
    def __init__(self):
        self._handlers: dict[str, List[Callable]] = {}

    def on(self, event: str, handler: Callable):
        self._handlers.setdefault(event, []).append(handler)
        return self   # fluent interface

    def emit(self, event: str, *args, **kwargs):
        for handler in self._handlers.get(event, []):
            handler(*args, **kwargs)

emitter = EventEmitter()
emitter.on("login", lambda user: print(f"Log: {user} logged in"))
emitter.on("login", lambda user: print(f"Welcome email sent to {user}"))
emitter.on("logout", lambda user: print(f"Goodbye {user}"))

emitter.emit("login", "Alice")
emitter.emit("logout", "Alice")

In [ ]:
# 3. Strategy Pattern — functions as strategies
from typing import Callable

class DataProcessor:
    def __init__(self, validate: Callable, transform: Callable, export: Callable):
        self.validate = validate
        self.transform = transform
        self.export = export

    def run(self, data):
        self.validate(data)
        result = self.transform(data)
        self.export(result)

def validate_not_empty(data):
    if not data:
        raise ValueError("Empty data")

def normalize(data):
    total = sum(data)
    return [x / total for x in data]

def print_export(data):
    print("Exported:", [f"{x:.3f}" for x in data])

processor = DataProcessor(validate_not_empty, normalize, print_export)
processor.run([10, 20, 30, 40])

In [ ]:
# 4. Context Manager as Template Method
from contextlib import contextmanager

@contextmanager
def transaction(conn_string):
    """Template: BEGIN → yield → COMMIT (or ROLLBACK on error)"""
    print(f"BEGIN [{conn_string}]")
    try:
        yield {"connection": conn_string}
        print("COMMIT")
    except Exception as e:
        print(f"ROLLBACK — {e}")
        raise

with transaction("postgresql://localhost/mydb") as tx:
    print(f"  Executing queries on {tx['connection']}")

## 9.4 Testing Best Practices — TDD, Fixtures, Mocking

### Concept

**Test-Driven Development (TDD)**: write failing test → write minimal code → refactor.  
**Fixtures** set up reusable test state. **Mocking** replaces real dependencies with controllable fakes.

In [ ]:
# TDD Example — building a shopping cart test-first
import pytest
from dataclasses import dataclass, field
from typing import List

@dataclass
class CartItem:
    name: str
    price: float
    quantity: int = 1

@dataclass
class Cart:
    items: List[CartItem] = field(default_factory=list)
    discount: float = 0.0  # 0.0 to 1.0

    def add(self, item: CartItem):
        self.items.append(item)

    def subtotal(self) -> float:
        return sum(i.price * i.quantity for i in self.items)

    def total(self) -> float:
        return self.subtotal() * (1 - self.discount)

    def item_count(self) -> int:
        return sum(i.quantity for i in self.items)


# pytest fixtures
@pytest.fixture
def empty_cart():
    return Cart()

@pytest.fixture
def cart_with_items():
    cart = Cart(discount=0.10)
    cart.add(CartItem("Apple", 1.50, 3))
    cart.add(CartItem("Bread", 2.50, 1))
    return cart


# Tests
def test_empty_cart_total(empty_cart):
    assert empty_cart.total() == 0.0

def test_subtotal(cart_with_items):
    # 3*1.50 + 2.50 = 7.00
    assert cart_with_items.subtotal() == pytest.approx(7.00)

def test_total_with_discount(cart_with_items):
    # 7.00 * 0.90 = 6.30
    assert cart_with_items.total() == pytest.approx(6.30)

def test_item_count(cart_with_items):
    assert cart_with_items.item_count() == 4  # 3 + 1


# Run the tests
import subprocess, sys
# In a real project: pytest test_cart.py -v
# Here we run programmatically:
cart = Cart(discount=0.10)
cart.add(CartItem("Apple", 1.50, 3))
cart.add(CartItem("Bread", 2.50, 1))
assert cart.subtotal() == 7.00
assert abs(cart.total() - 6.30) < 1e-9
print("All cart tests passed")

## 9.5 Packaging and Distribution

### Concept

Modern Python packages use `pyproject.toml` (PEP 517/518) as the single source of truth for build configuration.  
Tools: `pip`, `build`, `twine` (upload to PyPI), `hatch`, `poetry`, `uv`.

In [ ]:
# pyproject.toml example — shown as a string
PYPROJECT_TOML = """
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my-package"
version = "0.1.0"
description = "A sample Python package"
requires-python = ">=3.11"
dependencies = [
    "pydantic>=2.0",
    "httpx>=0.26",
]

[project.optional-dependencies]
dev = ["pytest", "ruff", "mypy"]

[project.scripts]
my-tool = "my_package.cli:main"

[tool.ruff]
line-length = 88
select = ["E", "F", "I"]

[tool.mypy]
strict = true
"""

print("Standard project structure:")
print("""
my-package/
├── src/
│   └── my_package/
│       ├── __init__.py
│       ├── core.py
│       └── cli.py
├── tests/
│   ├── conftest.py
│   └── test_core.py
├── pyproject.toml
└── README.md
""")

### Exercises — Section 9

1. Refactor this imperative code to be Pythonic: a loop that builds a dict of `{word: count}` from a text string, filtering words shorter than 4 characters.
2. Profile `sum(i**2 for i in range(100_000))` vs `sum([i**2 for i in range(100_000)])` using `timeit` and explain the difference.
3. Implement the **Command pattern**: create a `CommandHistory` class with `execute(cmd)`, `undo()` methods, and a `TextEditor` with `write(text)` and `delete(n)` commands that are undoable.

### Mini Challenge

Build a **dependency injection container** in under 50 lines: a `Container` class that can register factories with `.register(name, factory)` and resolve them with `.resolve(name)`, supporting transient (new instance each call) and singleton (cached) lifetimes.

### Solutions

In [ ]:
# Exercise 1 — Pythonic word count
# YOUR CODE HERE
# Exercise 3 — Command Pattern
# YOUR CODE HERE
# Mini Challenge — DI Container
class Container:
    # YOUR CODE HERE
    pass

---
# Section 10 — Real-World Case Studies

## 10.1 Case Study: Data Pipeline — CSV → Transform → JSON

### Overview

Build a production-quality ETL pipeline that:
1. Reads CSV data
2. Validates and cleans records
3. Transforms and enriches data
4. Writes to JSON output
5. Logs progress and errors

In [ ]:
import csv
import json
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterator, Optional

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)


@dataclass
class RawRecord:
    name: str
    age: str
    salary: str
    department: str


@dataclass
class CleanRecord:
    name: str
    age: int
    salary: float
    department: str
    salary_band: str


def read_csv(path: Path) -> Iterator[RawRecord]:
    with path.open() as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield RawRecord(**{k: v.strip() for k, v in row.items()})


def validate(record: RawRecord) -> Optional[CleanRecord]:
    try:
        age = int(record.age)
        salary = float(record.salary)
        if not (18 <= age <= 80):
            raise ValueError(f"Invalid age: {age}")
        if salary < 0:
            raise ValueError(f"Negative salary: {salary}")
        return CleanRecord(
            name=record.name.title(),
            age=age,
            salary=salary,
            department=record.department.upper(),
            salary_band=(
                "junior" if salary < 50_000 else
                "mid"    if salary < 100_000 else
                "senior"
            ),
        )
    except (ValueError, KeyError) as e:
        log.warning("Skipping record %r: %s", record.name, e)
        return None


def run_pipeline(input_path: Path, output_path: Path) -> dict:
    log.info("Starting pipeline: %s", input_path)
    records = []
    errors = 0

    for raw in read_csv(input_path):
        clean = validate(raw)
        if clean:
            records.append(asdict(clean))
        else:
            errors += 1

    output = {
        "count": len(records),
        "errors": errors,
        "records": records,
    }
    output_path.write_text(json.dumps(output, indent=2))
    log.info("Done: %d records written, %d errors", len(records), errors)
    return output


# Create sample data
sample_csv = Path("/tmp/employees.csv")
sample_csv.write_text(
    "name,age,salary,department\n"
    "alice,30,75000,engineering\n"
    "bob,25,45000,hr\n"
    "carol,999,120000,engineering\n"  # invalid age
    "dave,35,-5000,finance\n"         # negative salary
    "eve,28,95000,product\n"
)

result = run_pipeline(sample_csv, Path("/tmp/employees.json"))
print(json.dumps(result, indent=2))

## 10.2 Case Study: REST API Client with `httpx`

### Overview

Build a typed, async REST API client with retries, authentication, and response validation using `httpx` and `pydantic`.

In [ ]:
# Sync client (works without httpx installed using stdlib)
import urllib.request
import json
from dataclasses import dataclass
from typing import Optional


@dataclass
class Post:
    id: int
    title: str
    body: str
    userId: int


class JSONPlaceholderClient:
    BASE_URL = "https://jsonplaceholder.typicode.com"

    def __init__(self, timeout: int = 10):
        self.timeout = timeout

    def _get(self, endpoint: str) -> dict:
        url = f"{self.BASE_URL}{endpoint}"
        with urllib.request.urlopen(url, timeout=self.timeout) as resp:
            return json.loads(resp.read())

    def get_post(self, post_id: int) -> Post:
        data = self._get(f"/posts/{post_id}")
        return Post(**data)

    def get_posts(self, limit: int = 5) -> list[Post]:
        data = self._get(f"/posts?_limit={limit}")
        return [Post(**p) for p in data]


client = JSONPlaceholderClient()
try:
    post = client.get_post(1)
    print(f"Post #{post.id}: {post.title[:50]}")

    posts = client.get_posts(3)
    for p in posts:
        print(f"  [{p.userId}] {p.title[:40]}")
except Exception as e:
    print(f"Network unavailable: {e}")
    # Mock output for offline environments:
    print("Post #1: sunt aut facere repellat provident...")

## 10.3 Case Study: CLI Tool with `argparse` + `logging`

### Overview

A production CLI tool that processes files, supports multiple output formats, and has structured logging.

In [ ]:
import argparse
import json
import csv
import logging
import sys
from pathlib import Path


def setup_logging(verbose: bool) -> None:
    level = logging.DEBUG if verbose else logging.INFO
    logging.basicConfig(level=level, format="%(asctime)s [%(levelname)s] %(message)s")


def analyze_csv(path: Path) -> dict:
    rows = []
    with path.open() as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)

    headers = list(rows[0].keys()) if rows else []
    return {
        "file":    str(path),
        "rows":    len(rows),
        "columns": len(headers),
        "headers": headers,
    }


def main(args=None) -> int:
    parser = argparse.ArgumentParser(prog="csv-analyzer", description="Analyze CSV files")
    parser.add_argument("files", nargs="+", type=Path, help="CSV files to analyze")
    parser.add_argument("-f", "--format", choices=["json", "text"], default="text")
    parser.add_argument("-v", "--verbose", action="store_true")
    parser.add_argument("-o", "--output", type=Path, help="Output file (default: stdout)")
    ns = parser.parse_args(args)

    setup_logging(ns.verbose)
    log = logging.getLogger("csv-analyzer")

    results = []
    for path in ns.files:
        if not path.exists():
            log.error("File not found: %s", path)
            continue
        log.debug("Analyzing %s", path)
        results.append(analyze_csv(path))

    if ns.format == "json":
        output = json.dumps(results, indent=2)
    else:
        lines = []
        for r in results:
            lines.append(f"File: {r['file']} | Rows: {r['rows']} | Cols: {r['columns']}")
        output = "\n".join(lines)

    if ns.output:
        ns.output.write_text(output)
        log.info("Written to %s", ns.output)
    else:
        print(output)

    return 0


# Demo in notebook — simulate CLI args
rc = main([str(sample_csv), "--format", "json", "--verbose"])
print(f"Exit code: {rc}")

## 10.4 Case Study: Async Web Scraper with `asyncio`

### Overview

Scrape multiple URLs concurrently using `asyncio` and `urllib` (stdlib). Shows rate limiting, error handling, and result aggregation.

In [ ]:
import asyncio
import time
from dataclasses import dataclass
from typing import Optional


@dataclass
class ScrapedPage:
    url: str
    status: int
    size: int
    elapsed: float
    error: Optional[str] = None


# Simulate fetching (using asyncio.sleep as stand-in for real I/O)
async def fetch_page(session_id: int, url: str, semaphore: asyncio.Semaphore) -> ScrapedPage:
    async with semaphore:   # rate limit: max N concurrent requests
        start = time.perf_counter()
        try:
            # Simulate variable network latency
            import random
            delay = random.uniform(0.05, 0.3)
            await asyncio.sleep(delay)

            if "error" in url:
                raise ConnectionError("Simulated connection error")

            return ScrapedPage(
                url=url,
                status=200,
                size=random.randint(1000, 50000),
                elapsed=time.perf_counter() - start,
            )
        except Exception as e:
            return ScrapedPage(
                url=url,
                status=0,
                size=0,
                elapsed=time.perf_counter() - start,
                error=str(e),
            )


async def scrape_all(urls: list[str], max_concurrent: int = 5) -> list[ScrapedPage]:
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [
        asyncio.create_task(fetch_page(i, url, semaphore))
        for i, url in enumerate(urls)
    ]
    return await asyncio.gather(*tasks)


urls = [
    "https://example.com/page1",
    "https://example.com/page2",
    "https://example.com/page3",
    "https://error.example.com/broken",
    "https://example.com/page4",
    "https://example.com/page5",
]

start = time.perf_counter()
pages = asyncio.run(scrape_all(urls, max_concurrent=3))
total_time = time.perf_counter() - start

print(f"Scraped {len(pages)} pages in {total_time:.2f}s")
print()
for p in pages:
    if p.error:
        print(f"  FAIL  {p.url}: {p.error}")
    else:
        print(f"  OK    {p.url} | {p.size:>6} bytes | {p.elapsed:.3f}s")

successful = [p for p in pages if not p.error]
print(f"\nSuccess: {len(successful)}/{len(pages)}")
print(f"Avg size: {sum(p.size for p in successful)/len(successful):.0f} bytes")

## 10.5 Final Project: Full Python Application

### Overview

A complete, self-contained application combining:
- Typed data models (`dataclass`)
- Repository pattern for data access
- Service layer with business logic
- CLI interface
- Structured logging
- Unit tests

In [ ]:
# Task Manager — complete application
from __future__ import annotations
import json
import logging
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from enum import Enum

log = logging.getLogger("taskmanager")


class Priority(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


@dataclass
class Task:
    id: int
    title: str
    done: bool = False
    priority: Priority = Priority.MEDIUM
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    tags: list[str] = field(default_factory=list)


class TaskRepository:
    def __init__(self, storage_path: Path):
        self._path = storage_path
        self._tasks: dict[int, Task] = {}
        self._next_id = 1
        self._load()

    def _load(self):
        if self._path.exists():
            data = json.loads(self._path.read_text())
            for t in data.get("tasks", []):
                t["priority"] = Priority(t["priority"])
                task = Task(**t)
                self._tasks[task.id] = task
            self._next_id = max((t.id for t in self._tasks.values()), default=0) + 1

    def _save(self):
        data = {"tasks": [{**asdict(t), "priority": t.priority.value} for t in self._tasks.values()]}
        self._path.write_text(json.dumps(data, indent=2))

    def add(self, title: str, priority: Priority = Priority.MEDIUM, tags: list[str] = None) -> Task:
        task = Task(id=self._next_id, title=title, priority=priority, tags=tags or [])
        self._tasks[task.id] = task
        self._next_id += 1
        self._save()
        log.info("Added task #%d: %s", task.id, task.title)
        return task

    def complete(self, task_id: int) -> Task:
        if task_id not in self._tasks:
            raise KeyError(f"Task #{task_id} not found")
        self._tasks[task_id].done = True
        self._save()
        return self._tasks[task_id]

    def delete(self, task_id: int) -> None:
        if task_id not in self._tasks:
            raise KeyError(f"Task #{task_id} not found")
        del self._tasks[task_id]
        self._save()

    def list_tasks(self, done: Optional[bool] = None, tag: Optional[str] = None) -> list[Task]:
        tasks = list(self._tasks.values())
        if done is not None:
            tasks = [t for t in tasks if t.done == done]
        if tag:
            tasks = [t for t in tasks if tag in t.tags]
        return sorted(tasks, key=lambda t: (t.done, t.priority.value))

    def stats(self) -> dict:
        total = len(self._tasks)
        done  = sum(1 for t in self._tasks.values() if t.done)
        by_priority = {p.value: sum(1 for t in self._tasks.values() if t.priority == p)
                       for p in Priority}
        return {"total": total, "done": done, "pending": total - done, "by_priority": by_priority}


# Demo
repo = TaskRepository(Path("/tmp/tasks.json"))

t1 = repo.add("Write project docs", Priority.HIGH, tags=["docs", "urgent"])
t2 = repo.add("Fix login bug", Priority.HIGH, tags=["bug"])
t3 = repo.add("Refactor data layer", Priority.MEDIUM, tags=["refactor"])
t4 = repo.add("Update README", Priority.LOW, tags=["docs"])

repo.complete(t1.id)

print("=== All Tasks ===")
for task in repo.list_tasks():
    status = "[x]" if task.done else "[ ]"
    print(f"  {status} #{task.id} [{task.priority.value:6}] {task.title} {task.tags}")

print("\n=== Pending tasks ===")
for task in repo.list_tasks(done=False):
    print(f"  #{task.id} {task.title}")

print("\n=== Stats ===")
print(json.dumps(repo.stats(), indent=2))

print("\n=== Docs tag ===")
for task in repo.list_tasks(tag="docs"):
    print(f"  #{task.id} {task.title}")

### Final Challenge

Extend the Task Manager with:

1. **Due dates**: Add a `due_date: Optional[date]` field. Add a method `overdue()` that returns all undone tasks past their due date.
2. **Async notifications**: Write an `async def notify_overdue(repo, notifier)` function that checks for overdue tasks and calls `await notifier.send(task)` for each one.
3. **CLI integration**: Wire the `TaskRepository` to the `argparse` CLI with subcommands: `add`, `done`, `list`, `delete`, `stats`.
4. **Full test suite**: Write `pytest` tests covering: add, complete, delete, list filtering, stats, and error cases.

### Summary — Section 10

Real-world Python applications combine:

| Concern | Tool/Pattern |
|---------|-------------|
| Data modeling | `@dataclass`, `pydantic` |
| Persistence | JSON/CSV/pickle, databases |
| Business logic | Service layer, domain objects |
| Concurrency | `asyncio` for I/O, `ProcessPoolExecutor` for CPU |
| CLI | `argparse` |
| Logging | Structured `logging` |
| Testing | `pytest` + fixtures + mocks |
| Type safety | `typing` + `mypy` |

---

## Course Complete

You have covered the full Python curriculum from first `print("Hello")` to production-grade async applications.

**Recommended next steps:**
- Dive into domain-specific libraries: `FastAPI` (web), `PySpark` (data engineering), `PyTorch` (ML)
- Practice on real projects — apply every pattern from this course
- Contribute to open source — reading production Python is the fastest way to level up

---
*Python Course — Part 3 | Sections 7-10 | Complete*